In [1]:
import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/bearlifter/Data-Mining/main/Quirofano_Datos/playstore_dataset.csv"
df = pd.read_csv(url)

print("Registros:", df.shape[0])
print("Columnas:", df.shape[1])
df.head()

Registros: 10837
Columnas: 13


,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,"10,000+",Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,"5,000,000+",Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25M,"50,000,000+",Free,0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8M,"100,000+",Free,0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up


In [2]:
df.dtypes

App                object
Category           object
Rating            float64
Reviews             int64
Size               object
Installs           object
Type               object
Price              object
Content Rating     object
Genres             object
Last Updated       object
Current Ver        object
Android Ver        object
dtype: object

# Reto 1 Trampa de las unidades de medida
Escribe una función pura en Python que reciba un string. Si el string termina en 'M', quítale la 'M' y conviértelo a flotante (dejándolo como Megabytes). Si termina en 'k', quítale la 'k', conviértelo a flotante y divídelo entre 1024 (para pasarlo también a Megabytes). Si dice "Varies with device", devuélvelo como un valor nulo de Numpy (np.nan).

Aplica esta función a toda la columna utilizando el método .apply().

In [3]:
print("Valores nulos:", df['Size'].isnull().sum())
print("Terminan en 'M':", df['Size'].dropna().str.endswith('M').sum())
print("Terminan en 'k':", df['Size'].dropna().str.endswith('k').sum())
print()
print("Ejemplos:", df['Size'].dropna().unique()[:6])

Valores nulos: 1694
Terminan en 'M': 8827
Terminan en 'k': 316

Ejemplos: ['19M' '14M' '8.7M' '25M' '2.8M' '5.6M']


In [4]:
def convertir_size(valor):


    if pd.isna(valor):
        return np.nan

    texto = str(valor).strip()

    if texto == 'Varies with device':
        return np.nan

    if texto.endswith('M'):
        return float(texto[:-1])

    if texto.endswith('k'):
        return float(texto[:-1]) / 1024


    return np.nan

In [5]:
for prueba in ['19M', '201k', 'Varies with device', np.nan]:
    print(prueba, '->', convertir_size(prueba))

19M -> 19.0
201k -> 0.1962890625
Varies with device -> nan
nan -> nan


In [6]:

df['Size'] = df['Size'].apply(convertir_size)

print("Tipo de dato nuevo:", df['Size'].dtype)
df['Size'].head()

Tipo de dato nuevo: float64


0    19.0
1    14.0
2     8.7
3    25.0
4     2.8
Name: Size, dtype: float64

In [7]:
print("Peso promedio:", round(df['Size'].mean(), 2), "MB")
print("Mediana:      ", round(df['Size'].median(), 2), "MB")
print("Mínimo:       ", df['Size'].min(), "MB")
print("Máximo:       ", df['Size'].max(), "MB")

Peso promedio: 21.52 MB
Mediana:       13.0 MB
Mínimo:        0.00830078125 MB
Máximo:        100.0 MB


El peso promedio de las apps es de 21.52mb

# Reto 2: El tipo de dato cronologico
Investiga y utiliza la función pd.to_datetime() de Pandas para sobrescribir la columna Last Updated, convirtiéndola del tipo string (Object) al tipo nativo datetime64.
Ahora que es un objeto de tiempo, Pandas te permite extraer componentes específicos. Crea una nueva columna llamada Year_Updated extrayendo únicamente el año (df['Last Updated'].dt.year).
Pregunta a responder: Utilizando la sumarización categórica (value_counts()) sobre tu nueva columna Year_Updated, ¿en qué año se actualizó la mayor cantidad de aplicaciones en nuestro dataset?

In [8]:
print("Tipo actual:", df['Last Updated'].dtype)
df['Last Updated'].head()

Tipo actual: object


0     January 7, 2018
1    January 15, 2018
2      August 1, 2018
3        June 8, 2018
4       June 20, 2018
Name: Last Updated, dtype: object

In [9]:
df['Last Updated'] = pd.to_datetime(df['Last Updated'], format='%B %d, %Y')

print("Tipo nuevo:", df['Last Updated'].dtype)
print("Fecha más antigua:", df['Last Updated'].min().date())
print("Fecha más reciente:", df['Last Updated'].max().date())

Tipo nuevo: datetime64[ns]
Fecha más antigua: 2010-05-21
Fecha más reciente: 2018-08-08


In [10]:
df['Year_Updated'] = df['Last Updated'].dt.year

df[['App', 'Last Updated', 'Year_Updated']].head()

,App,Last Updated,Year_Updated
0,Photo Editor & Candy Camera & Grid & ScrapBook,2018-01-07,2018
1,Coloring book moana,2018-01-15,2018
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",2018-08-01,2018
3,Sketch - Draw & Paint,2018-06-08,2018
4,Pixel Draw - Number Art Coloring Book,2018-06-20,2018


In [11]:
conteo_anios = df['Year_Updated'].value_counts()
conteo_anios.head(8)

2018    7346
2017    1867
2016     804
2015     459
2014     209
2013     110
2012      26
2011      15
Name: Year_Updated, dtype: int64

In [12]:
anio_top = conteo_anios.index[0]
print(f"Año con más actualizaciones: {anio_top}")
print(f"Apps actualizadas ese año: {conteo_anios.iloc[0]:,}")
print(f"Porcentaje del dataset: {conteo_anios.iloc[0] / len(df) * 100:.1f}%")

Año con más actualizaciones: 2018
Apps actualizadas ese año: 7,346
Porcentaje del dataset: 67.8%


# Respuesta:
El año con mas actualizaciones es 2018, con 7346 apps. Es decir, 67.8% de todo el dataset.

# Reto 3:
Tu misión algorítmica: Evalúa cuántos registros quedaron vacíos. Como ingenieros, decidan y apliquen la mejor técnica: ¿es matemáticamente más sano borrar esas filas con un .dropna() porque el peso de la app es crítico, o prefieren rellenar ese hueco imputando la mediana global del peso de las apps?
Pregunta a responder: Redacten una breve justificación técnica de 3 líneas explicando qué método eligieron y por qué lo consideran superior para no dañar al modelo de predicción.

In [13]:
faltantes = df['Size'].isnull().sum()

print(f"Apps sin tamaño declarado: {faltantes:,}")
print(f"Representan el {faltantes / len(df) * 100:.2f}% del dataset")
print(f"Si las borráramos quedaríamos con {len(df) - faltantes:,} registros")

Apps sin tamaño declarado: 1,694
Representan el 15.63% del dataset
Si las borráramos quedaríamos con 9,143 registros


In [14]:
sin_tamanio = df['Size'].isnull()


installs_num = pd.to_numeric(
    df['Installs'].str.replace('+', '', regex=False).str.replace(',', '', regex=False),
    errors='coerce'
)

comparacion = pd.DataFrame({
    'Sin tamaño declarado': [
        installs_num[sin_tamanio].median(),
        df.loc[sin_tamanio, 'Reviews'].median(),
        df.loc[sin_tamanio, 'Rating'].median()
    ],
    'Con tamaño declarado': [
        installs_num[~sin_tamanio].median(),
        df.loc[~sin_tamanio, 'Reviews'].median(),
        df.loc[~sin_tamanio, 'Rating'].median()
    ]
}, index=['Instalaciones (mediana)', 'Reviews (mediana)', 'Rating (mediana)'])

comparacion

,Sin tamaño declarado,Con tamaño declarado
Instalaciones (mediana),10000000.0,100000.0
Reviews (mediana),107749.5,742.0
Rating (mediana),4.3,4.3


In [15]:

mediana_global = df['Size'].median()


df['Size_Imputado'] = df['Size'].isnull()

df['Size'] = df['Size'].fillna(mediana_global)

print(f"Mediana usada para imputar: {mediana_global} MB")
print(f"Valores imputados: {df['Size_Imputado'].sum():,}")
print(f"Nulos restantes en Size: {df['Size'].isnull().sum()}")
print(f"Registros conservados: {len(df):,}")

Mediana usada para imputar: 13.0 MB
Valores imputados: 1,694
Nulos restantes en Size: 0
Registros conservados: 10,837


# Justificacion tecnica
Elegimos imputar con la mediana global en lugar de eliminar las filas. La razón es que el faltante no es
aleatorio: las apps sin tamaño declarado tienen una mediana de 10 millones de instalaciones contra 100
mil del resto, o sea que un dropna() no quitaría un 15% cualquiera del dataset, sino específicamente
a las apps más exitosas del catálogo.

Usamos la mediana y no el promedio porque la distribución está sesgada por las apps muy pesadas, y
agregamos la columna Size_Imputado como bandera para que el modelo pueda distinguir el dato real del
rellenado.